# v5: Conservative Hybrid Deep Research Agent

v4 的运行结果说明：更多检索和更多文档窗口并不必然提升准确率，反而会让模型被错误候选带偏。v5 改为保守混合策略：先运行 compact evidence path；只有 compact 答案弱、无法回答或低置信时，才启用 planner expansion path。

本 notebook 只定义方案和代码。本地不要执行、不要安装依赖、不要构建索引、不要启动或调用 vLLM。运行轨迹额外写入 `./trace/v5_trace.jsonl`。

## 1. 配置与导入

v5 复用 v3/v4 已有工具集合，不新增外部依赖，不替换 BM25 检索器。

In [ ]:
from pathlib import Path
import json
import re
import sys
from typing import Any, Dict, List, Optional

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from agent.dataset_utils import load_jsonl
from agent.eval import run_evaluation
from agent.tools import build_searcher, get_v3_research_tool_specs_and_registry, make_initial_search_queries
from agent.vllm_client import VLLMClient

VLLM_BASE_URL = "http://127.0.0.1:8000/v1"
MODEL_NAME = "qwen_auto"
API_KEY = "dummy"

BM25_INDEX_PATH = str(project_root / "indexes" / "browsecomp_plus_bm25.sqlite")
HARD50_PATH = str(project_root / "browsecomp_plus_hard50.jsonl")
SUBMISSION_PATH = str(project_root / "runs" / "v5_submission.jsonl")
EVAL_OUTPUT_PATH = str(project_root / "runs" / "v5_eval_results.jsonl")
TRACE_DIR = project_root / "trace"
TRACE_PATH = str(TRACE_DIR / "v5_trace.jsonl")

client = VLLMClient(base_url=VLLM_BASE_URL, api_key=API_KEY)
searcher = build_searcher(index_path=BM25_INDEX_PATH)
tool_specs, tool_registry = get_v3_research_tool_specs_and_registry(
    searcher=searcher,
    k=8,
    snippet_max_chars=1400,
    document_window_chars=2400,
)


## 2. Prompt

v5 明确恢复保守策略：证据不足时允许 `Unable to determine`，避免 v4 因强迫猜测导致错误具体答案。

In [ ]:
COMPACT_ANSWER_PROMPT = """
You are a conservative evidence-grounded QA agent for BrowseComp-Plus.
Use only the retrieved local corpus evidence in the conversation.

Rules:
1. Do not answer from memory.
2. If the evidence directly supports a short answer, provide it.
3. If evidence is weak, conflicting, or only loosely related, return 'Unable to determine' with low confidence.
4. Do not overfit to a familiar entity unless it satisfies the question's constraints.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()

PLANNER_PROMPT = """
Generate a compact BM25 search expansion plan. Do not answer the question.
Return JSON only with keys search_queries, key_phrases, keywords, answer_type.
Queries should be short, literal, and evidence-seeking.
""".strip()

ADJUDICATOR_PROMPT = """
You choose the safer final answer between two candidate answers for BrowseComp-Plus.
Use only the provided evidence summaries and candidate outputs.

Rules:
1. If one candidate is concrete and well-supported while the other is unsupported, choose the supported one.
2. If candidates conflict and neither is directly supported, choose 'Unable to determine'.
3. Do not invent a third answer unless it is explicitly stated in the evidence.
4. Output the required final answer format.

Return exactly:
Explanation: ...
Evidence: ...
Exact Answer: ...
Confidence: ...%
""".strip()


## 3. 通用辅助函数

包括答案抽取、置信度解析、低质量答案判断、轻量重排和 trace 压缩。

In [ ]:
STOPWORDS = {
    "the", "and", "for", "that", "with", "from", "this", "what", "which", "who", "whose", "where",
    "when", "about", "into", "between", "after", "before", "during", "their", "there", "would",
    "could", "should", "please", "provide", "identify", "looking", "question", "answer", "according",
}


def canonical_text(text: str) -> str:
    return re.sub(r"\s+", " ", str(text or "").strip().lower())


def truncate_text(text: str, max_chars: int) -> str:
    text = str(text or "")
    if len(text) <= max_chars:
        return text
    return text[:max_chars].rstrip() + "... [truncated]"


def strip_thinking(text: str) -> str:
    text = str(text or "")
    text = re.sub(r"(?is)<think>.*?</think>\s*", "", text).strip()
    text = re.sub(r"(?is)<think>.*", "", text).strip()
    return text


def extract_exact_answer(text: str) -> str:
    original = str(text or "").strip()
    clean = strip_thinking(original)
    for candidate_text in (clean, original):
        match = re.search(r"(?im)^\s*Exact Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return match.group(1).strip()
        match = re.search(r"(?im)^\s*Answer\s*:\s*(.+?)\s*$", candidate_text)
        if match:
            return match.group(1).strip()
    if clean and len(clean) <= 160:
        return clean
    return "Unable to determine"


def extract_confidence(text: str, default: int = 0) -> int:
    match = re.search(r"(?im)^\s*Confidence\s*:\s*(\d{1,3})\s*%", str(text or ""))
    if not match:
        return default
    return max(0, min(100, int(match.group(1))))


def is_unable_answer(answer: str) -> bool:
    return bool(re.search(r"(?i)unable|cannot|can't|not enough|insufficient|not determine|not specified|unknown", str(answer or "")))


def should_expand(candidate: Dict[str, Any], min_confidence: int = 55) -> bool:
    answer = candidate.get("predicted_answer", "")
    confidence = int(candidate.get("confidence", 0) or 0)
    if is_unable_answer(answer):
        return True
    if confidence and confidence < min_confidence:
        return True
    if len(str(answer).strip()) == 0 or len(str(answer)) > 140:
        return True
    return False


def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    text = strip_thinking(text)
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None


def normalize_string_list(value: Any, limit: int = 10, max_chars: int = 160) -> List[str]:
    if isinstance(value, str):
        items = [value]
    elif isinstance(value, list):
        items = value
    else:
        items = []
    output = []
    seen = set()
    for item in items:
        text = " ".join(str(item or "").replace("\n", " ").split()).strip(" ,.;:")
        if len(text) > max_chars:
            text = text[:max_chars].rstrip()
        key = text.lower()
        if text and key not in seen:
            output.append(text)
            seen.add(key)
        if len(output) >= limit:
            break
    return output


def tokenize_for_score(text: str) -> List[str]:
    tokens = re.findall(r"[A-Za-z0-9][A-Za-z0-9_\-']+", str(text or "").lower())
    return [token.strip("'-") for token in tokens if len(token.strip("'-")) >= 4 and token not in STOPWORDS]


def rank_search_results(results: List[Dict[str, Any]], terms: List[str], limit: int = 8) -> List[Dict[str, Any]]:
    ranked = []
    for item in results:
        haystack = canonical_text(" ".join([item.get("query", ""), item.get("snippet", ""), item.get("url", "")]))
        overlap = sum(1 for term in terms if term in haystack)
        score = float(item.get("score", 0) or 0)
        ranked.append({**item, "controller_score": overlap * 1000 + score, "term_overlap": overlap})
    ranked.sort(key=lambda row: (row.get("term_overlap", 0), row.get("score", 0)), reverse=True)
    return ranked[:limit]


def compact_result_summary(results: List[Dict[str, Any]], max_items: int = 8) -> List[Dict[str, Any]]:
    compact = []
    for item in results[:max_items]:
        compact.append(
            {
                "docid": item.get("docid", ""),
                "score": item.get("score", 0),
                "controller_score": item.get("controller_score", 0),
                "url": item.get("url", ""),
                "query": item.get("query", ""),
                "snippet": truncate_text(item.get("snippet", ""), 450),
            }
        )
    return compact


## 4. 工具执行与轨迹状态

控制器调用仍写入 `messages`，同时 v5 额外生成 trace record，便于后续分析每条样本选择 compact 还是 expanded。

In [ ]:
def normalize_tool_result(tool_name: str, result: Any, max_document_chars: int = 4200) -> Any:
    if tool_name == "get_document" and isinstance(result, dict):
        normalized = dict(result)
        if "text" in normalized:
            full_text = str(normalized.get("text", ""))
            normalized["text"] = truncate_text(full_text, max_document_chars)
            normalized["text_length"] = len(full_text)
            normalized["text_is_truncated"] = len(full_text) > max_document_chars
        return normalized
    return result


def execute_tool(tool_name: str, arguments: Dict[str, Any]) -> Dict[str, Any]:
    if tool_name not in tool_registry:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": "unknown tool"}
    try:
        raw_result = tool_registry[tool_name](**arguments)
        return {
            "ok": True,
            "tool_name": tool_name,
            "arguments": arguments,
            "tool_result": normalize_tool_result(tool_name, raw_result),
        }
    except Exception as exc:
        return {"ok": False, "tool_name": tool_name, "arguments": arguments, "error": repr(exc)}


def make_tool_message(tool_call_id: str, executed: Dict[str, Any]) -> Dict[str, Any]:
    content = executed.get("tool_result") if executed.get("ok") else executed
    return {"role": "tool", "tool_call_id": tool_call_id, "content": json.dumps(content, ensure_ascii=False)}


def docids_from_tool_result(tool_name: str, result: Any) -> List[str]:
    if tool_name in {"search", "search_many"} and isinstance(result, list):
        return [str(item.get("docid", "")) for item in result if isinstance(item, dict) and item.get("docid")]
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return [str(item.get("docid", "")) for item in result.get("snippets", []) if item.get("docid")]
    if isinstance(result, dict) and result.get("docid"):
        return [str(result.get("docid"))]
    return []


def initial_state(question: str) -> Dict[str, Any]:
    return {
        "question": question,
        "mode": "compact",
        "search_queries": [],
        "seen_docids": [],
        "opened_docids": [],
        "ranked_results": [],
        "evidence_table": [],
        "tool_events": [],
        "candidate_answers": [],
        "decision": {},
    }


def summarize_tool_result(tool_name: str, result: Any) -> str:
    if tool_name in {"search", "search_many"}:
        return "docids=" + ", ".join(docids_from_tool_result(tool_name, result)[:12])
    if tool_name == "collect_evidence_snippets" and isinstance(result, dict):
        return f"collected {len(result.get('snippets', []))} snippets"
    if isinstance(result, dict) and result.get("docid"):
        return f"inspected docid={result.get('docid', '')}"
    return truncate_text(json.dumps(result, ensure_ascii=False), 300)


def update_state_from_execution(state: Dict[str, Any], executed: Dict[str, Any], round_id: int) -> None:
    tool_name = executed.get("tool_name", "")
    arguments = executed.get("arguments", {}) or {}
    result = executed.get("tool_result")
    if tool_name == "search_many":
        for query in arguments.get("queries", []) or []:
            if query and query not in state["search_queries"]:
                state["search_queries"].append(query)
    for docid in docids_from_tool_result(tool_name, result):
        if docid and docid not in state["seen_docids"]:
            state["seen_docids"].append(docid)
        if tool_name in {"get_document", "get_document_window", "find_in_document"} and docid not in state["opened_docids"]:
            state["opened_docids"].append(docid)
    if tool_name in {"find_in_document", "collect_evidence_snippets"}:
        state["evidence_table"].append(
            {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "summary": summarize_tool_result(tool_name, result)}
        )
    state["tool_events"].append(
        {"round_id": round_id, "tool_name": tool_name, "arguments": arguments, "ok": executed.get("ok", False), "summary": summarize_tool_result(tool_name, result)}
    )


def append_controller_tool_call(messages: List[Dict[str, Any]], state: Dict[str, Any], tool_name: str, arguments: Dict[str, Any], round_id: int, note: str) -> Dict[str, Any]:
    call_id = f"controller_{round_id}_{len(state['tool_events']) + 1}_{tool_name}"
    messages.append(
        {
            "role": "assistant",
            "content": note,
            "tool_calls": [
                {"id": call_id, "type": "function", "function": {"name": tool_name, "arguments": json.dumps(arguments, ensure_ascii=False)}}
            ],
        }
    )
    executed = execute_tool(tool_name, arguments)
    messages.append(make_tool_message(call_id, executed))
    update_state_from_execution(state, executed, round_id)
    return executed


def public_state_summary(state: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "mode": state["mode"],
        "search_queries": state["search_queries"][-12:],
        "seen_docids": state["seen_docids"][-35:],
        "opened_docids": state["opened_docids"][-16:],
        "ranked_results": compact_result_summary(state["ranked_results"], max_items=8),
        "evidence_table": state["evidence_table"][-10:],
        "candidate_answers": state["candidate_answers"],
        "decision": state["decision"],
    }


## 5. Compact Path 与 Expansion Path

compact path 类似 v3，但更保守；expansion path 类似 v4，但只在 compact 弱时触发，并且不强迫模型猜答案。

In [ ]:
def answer_from_state(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], prompt: str, max_tokens: int = 1100) -> Dict[str, Any]:
    state_message = {"role": "user", "content": "Compressed evidence state:\n" + json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)}
    final_instruction = {"role": "user", "content": prompt}
    response = client.simple_chat(
        model=MODEL_NAME,
        messages=[messages[0], messages[1], state_message] + messages[2:] + [final_instruction],
        temperature=0.0,
        max_tokens=max_tokens,
    )
    content = response["choices"][0]["message"].get("content", "")
    return {"content": content, "predicted_answer": extract_exact_answer(content), "confidence": extract_confidence(content)}


def run_compact_path(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], per_query_k: int = 5, auto_open_docs: int = 2) -> Dict[str, Any]:
    state["mode"] = "compact"
    queries = make_initial_search_queries(question, max_queries=4, max_query_chars=220)
    if not queries:
        queries = [question]
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": queries, "per_query_k": per_query_k},
        round_id=0,
        note="v5 compact path: initial BM25 searches.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question)
    ranked = rank_search_results(raw_results, terms=terms, limit=8)
    state["ranked_results"] = ranked
    messages.append({"role": "assistant", "content": "v5 compact ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})
    for item in ranked[:auto_open_docs]:
        docid = str(item.get("docid", ""))
        if docid:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 2400},
                round_id=0,
                note="v5 compact path: inspect compact candidate.",
            )
    candidate = answer_from_state(question, messages, state, COMPACT_ANSWER_PROMPT)
    candidate["mode"] = "compact"
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    return candidate


def plan_expansion(question: str) -> Dict[str, Any]:
    fallback = {"search_queries": make_initial_search_queries(question, max_queries=5, max_query_chars=180), "key_phrases": [], "keywords": tokenize_for_score(question)[:16], "answer_type": "short answer"}
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": PLANNER_PROMPT}, {"role": "user", "content": question}],
            temperature=0.0,
            max_tokens=800,
        )
        parsed = extract_json_object(response["choices"][0]["message"].get("content", ""))
    except Exception:
        parsed = None
    if not parsed:
        parsed = fallback
    planner_queries = normalize_string_list(parsed.get("search_queries"), limit=7, max_chars=150)
    deterministic = make_initial_search_queries(question, max_queries=4, max_query_chars=180)
    key_phrases = normalize_string_list(parsed.get("key_phrases"), limit=10, max_chars=100)
    keywords = normalize_string_list(parsed.get("keywords"), limit=14, max_chars=60)
    return {
        "search_queries": normalize_string_list(deterministic + planner_queries + key_phrases, limit=8, max_chars=150),
        "key_phrases": key_phrases,
        "keywords": keywords,
        "answer_type": str(parsed.get("answer_type", "short answer")),
    }


def run_expansion_path(question: str, messages: List[Dict[str, Any]], state: Dict[str, Any], per_query_k: int = 5, auto_open_docs: int = 3) -> Dict[str, Any]:
    state["mode"] = "expanded"
    plan = plan_expansion(question)
    messages.append({"role": "assistant", "content": "v5 expansion planner:\n" + json.dumps(plan, ensure_ascii=False, indent=2)})
    executed = append_controller_tool_call(
        messages,
        state,
        "search_many",
        {"queries": plan["search_queries"], "per_query_k": per_query_k},
        round_id=1,
        note="v5 expansion path: execute additional planned searches.",
    )
    raw_results = executed.get("tool_result", []) if executed.get("ok") else []
    terms = tokenize_for_score(question) + tokenize_for_score(" ".join(plan.get("keywords", []) + plan.get("key_phrases", [])))
    ranked = rank_search_results(raw_results, terms=terms, limit=10)
    state["ranked_results"] = ranked
    messages.append({"role": "assistant", "content": "v5 expansion ranked candidates:\n" + json.dumps(compact_result_summary(ranked), ensure_ascii=False, indent=2)})
    opened = 0
    for item in ranked:
        docid = str(item.get("docid", ""))
        if docid and docid not in state["opened_docids"]:
            append_controller_tool_call(
                messages,
                state,
                "get_document_window",
                {"docid": docid, "start": 0, "window_chars": 2400},
                round_id=1,
                note="v5 expansion path: inspect additional candidate.",
            )
            opened += 1
        if opened >= auto_open_docs:
            break
    keywords = ", ".join(normalize_string_list(plan.get("keywords") + plan.get("key_phrases"), limit=12, max_chars=80))
    top_docids = [str(item.get("docid", "")) for item in ranked[:8] if item.get("docid")]
    if keywords and top_docids:
        append_controller_tool_call(
            messages,
            state,
            "collect_evidence_snippets",
            {"docids": top_docids, "keywords": keywords, "window_chars": 850, "max_snippets": 10},
            round_id=1,
            note="v5 expansion path: collect additional evidence snippets.",
        )
    candidate = answer_from_state(question, messages, state, COMPACT_ANSWER_PROMPT)
    candidate["mode"] = "expanded"
    candidate["plan"] = plan
    state["candidate_answers"].append({k: candidate[k] for k in ("mode", "predicted_answer", "confidence")})
    return candidate


## 6. 候选答案选择与 v5 Agent

如果 compact 答案足够强，直接使用 compact；否则运行 expansion。若两者冲突，则让 adjudicator 保守选择。

In [ ]:
def adjudicate_candidates(question: str, state: Dict[str, Any], compact: Dict[str, Any], expanded: Dict[str, Any]) -> Dict[str, Any]:
    if not should_expand(compact):
        return {**compact, "selected_mode": "compact", "decision_reason": "compact_answer_confident_enough"}
    if should_expand(compact) and not should_expand(expanded):
        return {**expanded, "selected_mode": "expanded", "decision_reason": "compact_weak_expanded_concrete"}
    if is_unable_answer(compact.get("predicted_answer", "")) and is_unable_answer(expanded.get("predicted_answer", "")):
        return {**compact, "selected_mode": "compact", "decision_reason": "both_unable_keep_conservative"}

    evidence_summary = json.dumps(public_state_summary(state), ensure_ascii=False, indent=2)
    adjudication_input = {
        "role": "user",
        "content": (
            f"Question:\n{question}\n\n"
            f"Compact candidate:\n{compact.get('content', '')}\n\n"
            f"Expanded candidate:\n{expanded.get('content', '')}\n\n"
            f"Evidence summary:\n{evidence_summary}"
        ),
    }
    try:
        response = client.simple_chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": ADJUDICATOR_PROMPT}, adjudication_input],
            temperature=0.0,
            max_tokens=1000,
        )
        content = response["choices"][0]["message"].get("content", "")
        return {
            "mode": "adjudicated",
            "selected_mode": "adjudicated",
            "decision_reason": "adjudicator_used_for_conflict_or_weak_answers",
            "content": content,
            "predicted_answer": extract_exact_answer(content),
            "confidence": extract_confidence(content),
        }
    except Exception:
        if not is_unable_answer(expanded.get("predicted_answer", "")):
            return {**expanded, "selected_mode": "expanded", "decision_reason": "adjudicator_failed_keep_expanded"}
        return {**compact, "selected_mode": "compact", "decision_reason": "adjudicator_failed_keep_compact"}


def build_trace_record(row: Dict[str, Any], result: Dict[str, Any]) -> Dict[str, Any]:
    state = result["state_summary"]
    return {
        "query_id": str(row.get("query_id", "")),
        "status": result["status"],
        "predicted_answer": result["predicted_answer"],
        "decision": state.get("decision", {}),
        "candidate_answers": state.get("candidate_answers", []),
        "search_queries": state.get("search_queries", []),
        "seen_docids": state.get("seen_docids", []),
        "opened_docids": state.get("opened_docids", []),
        "evidence_table": state.get("evidence_table", []),
        "ranked_results": state.get("ranked_results", []),
    }


def run_v5_agent(
    question: str,
    query_id: Optional[str] = None,
    compact_per_query_k: int = 5,
    compact_auto_open_docs: int = 2,
    expansion_per_query_k: int = 5,
    expansion_auto_open_docs: int = 3,
) -> Dict[str, Any]:
    messages: List[Dict[str, Any]] = [
        {"role": "system", "content": COMPACT_ANSWER_PROMPT},
        {"role": "user", "content": question},
    ]
    state = initial_state(question)
    compact = run_compact_path(question, messages, state, per_query_k=compact_per_query_k, auto_open_docs=compact_auto_open_docs)

    expanded = None
    if should_expand(compact):
        expanded = run_expansion_path(question, messages, state, per_query_k=expansion_per_query_k, auto_open_docs=expansion_auto_open_docs)
        selected = adjudicate_candidates(question, state, compact, expanded)
    else:
        selected = {**compact, "selected_mode": "compact", "decision_reason": "compact_answer_confident_enough"}

    state["decision"] = {
        "selected_mode": selected.get("selected_mode", selected.get("mode", "unknown")),
        "decision_reason": selected.get("decision_reason", ""),
        "compact_predicted_answer": compact.get("predicted_answer", ""),
        "compact_confidence": compact.get("confidence", 0),
        "expanded_predicted_answer": expanded.get("predicted_answer", "") if expanded else None,
        "expanded_confidence": expanded.get("confidence", 0) if expanded else None,
    }

    messages.append({"role": "assistant", "content": selected.get("content", "")})
    return {
        "query_id": query_id,
        "status": "completed_v5_hybrid",
        "predicted_answer": selected.get("predicted_answer", ""),
        "final_output": selected.get("content", ""),
        "messages": messages,
        "state_summary": public_state_summary(state),
    }


## 7. Submission、Trace 与评测

`generate_submission()` 同时写 submission 和 `./trace/v5_trace.jsonl`，方便后续按 query_id 查看 compact/expanded/adjudicated 决策过程。

In [ ]:
def build_submission_record(row: Dict[str, Any], **agent_kwargs: Any) -> tuple[Dict[str, Any], Dict[str, Any]]:
    result = run_v5_agent(question=row["query"], query_id=str(row.get("query_id", "")), **agent_kwargs)
    record = {
        "query_id": str(row.get("query_id", "")),
        "predicted_answer": result["predicted_answer"],
        "status": result["status"],
        "messages": result["messages"],
        "state_summary": result["state_summary"],
    }
    trace = build_trace_record(row, result)
    return record, trace


def generate_submission(
    dataset_path: str = HARD50_PATH,
    output_path: str = SUBMISSION_PATH,
    trace_path: str = TRACE_PATH,
    limit: Optional[int] = 50,
    **agent_kwargs: Any,
) -> List[Dict[str, Any]]:
    rows = load_jsonl(dataset_path, limit=limit)
    output_file = Path(output_path)
    trace_file = Path(trace_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    trace_file.parent.mkdir(parents=True, exist_ok=True)
    records = []
    with output_file.open("w", encoding="utf-8") as fout, trace_file.open("w", encoding="utf-8") as trout:
        for index, row in enumerate(rows, start=1):
            print(f"[{index}/{len(rows)}] query_id={row.get('query_id', '')}")
            record, trace = build_submission_record(row, **agent_kwargs)
            records.append(record)
            fout.write(json.dumps(record, ensure_ascii=False) + "\n")
            trout.write(json.dumps(trace, ensure_ascii=False) + "\n")
    print(f"Saved {len(records)} records to {output_path}")
    print(f"Saved trace records to {trace_path}")
    return records


def evaluate_submission(
    submission_path: str = SUBMISSION_PATH,
    dataset_path: str = HARD50_PATH,
    output_path: str = EVAL_OUTPUT_PATH,
):
    return run_evaluation(
        submission_path=submission_path,
        dataset_path=dataset_path,
        model_name=MODEL_NAME,
        base_url=VLLM_BASE_URL,
        api_key=API_KEY,
        output_path=output_path,
        max_tokens=1024,
        verbose=True,
    )


## 8. 服务器运行入口

本地不要执行。到服务器确认 vLLM 和 BM25 索引已就绪后，再取消注释运行。

```python
# records = generate_submission(limit=50)
# summary, details = evaluate_submission()
```